In [ ]:
import kagglehub
path = kagglehub.dataset_download("adisongoh/it-service-ticket-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 3.45M/3.45M [00:00<00:00, 3.73MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/adisongoh/it-service-ticket-classification-dataset/versions/1


In [ ]:
import os
print(f"Files in {path}:")
for filename in os.listdir(path):
    print(filename)

Files in /root/.cache/kagglehub/datasets/adisongoh/it-service-ticket-classification-dataset/versions/1:
all_tickets_processed_improved_v3.csv


In [ ]:
import pandas as pd
csv_file_path = os.path.join(path, 'all_tickets_processed_improved_v3.csv')

df = pd.read_csv(csv_file_path)
print("DataFrame loaded successfully. Here's a preview:")
display(df.head())
print("\nDataFrame Info:")
df.info()

DataFrame loaded successfully. Here's a preview:


,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Document     47837 non-null  object
 1   Topic_group  47837 non-null  object
dtypes: object(2)
memory usage: 747.6+ KB


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

def spacy_clean(text):
    doc = nlp(str(text).lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

df['refined_text'] = df['Document'].apply(spacy_clean)

In [ ]:
def derive_priority(row):
    text = row['refined_text']
    cat = row['Topic_group']

    if any(word in text for word in ['critical', 'blocker', 'urgent', 'emergency']):
        return 'High'
    if cat in ['Access', 'Administrative rights']:
        return 'High'
    if cat in ['Hardware', 'Storage']:
        return 'Medium'
    return 'Low'

df['priority'] = df.apply(derive_priority, axis=1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['refined_text'], df['Topic_group'], test_size=0.2)

text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
    ('clf', LinearSVC(class_weight='balanced'))
])

text_clf.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('clf', LinearSVC(class_weight='balanced'))])

In [ ]:
from sklearn.metrics import classification_report

predictions = text_clf.predict(X_test)
print(classification_report(y_test, predictions))

                       precision    recall  f1-score   support

               Access       0.90      0.88      0.89      1442
Administrative rights       0.70      0.83      0.76       356
           HR Support       0.87      0.85      0.86      2158
             Hardware       0.85      0.81      0.83      2723
     Internal Project       0.80      0.88      0.84       427
        Miscellaneous       0.79      0.84      0.81      1413
             Purchase       0.91      0.92      0.92       500
              Storage       0.86      0.90      0.88       549

             accuracy                           0.85      9568
            macro avg       0.84      0.86      0.85      9568
         weighted avg       0.85      0.85      0.85      9568

